In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = ROOT / "dataset" / "kosovo_overall_biodiversity.csv"
REPORTS_DIR = ROOT / "reports"


def section(title):
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")

## 1. MBLEDHJA E TË DHËNAVE

In [2]:
section("1. MBLEDHJA E TË DHËNAVE")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Rreshta: {df_raw.shape[0]:,}  |  Kolona: {df_raw.shape[1]}")
print(df_raw.head(3).T)

df_raw.info(memory_usage="deep")


1. MBLEDHJA E TË DHËNAVE


Rreshta: 46,831  |  Kolona: 42
                                                                               0                                                  1                                                  2
gbifID                                                                6500474327                                         6500462703                                         6500450749
scientificName                                    Mantis religiosa (Linne, 1758)                  Issoria lathonia (Linnaeus, 1758)                  Podarcis muralis (Laurenti, 1768)
species                                                         Mantis religiosa                                   Issoria lathonia                                   Podarcis muralis
vernacularName                                                    Praying Mantis                          Queen of Spain Fritillary                                 Common Wall Lizard
kingdom                                               

## Madhësia e skedarëve

In [3]:
section("Madhësia e skedarëve")
GITHUB_LIMIT_MB = 100


def file_size_mb(path):
    return path.stat().st_size / 1024**2


print(f"Dataseti origjinal: {file_size_mb(DATA_PATH):.1f} MB")
print(f"Në memorie (pandas): {df_raw.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

drop_planned = ["iasAppliesFrom", "establishmentMeans", "identifiedBy", "iucnRedListStatus",
                "taxonKey", "catalogNumber", "collectionCode", "issue"]
cleaned_mb = len(df_raw.drop(columns=drop_planned).to_csv(index=False).encode("utf-8")) / 1024**2
sample_mb = len(df_raw.sample(frac=0.1, random_state=42).to_csv(index=False).encode("utf-8")) / 1024**2
print(f"Versioni i pastruar (vlerësim): {cleaned_mb:.1f} MB")
print(f"Mostra 10% (vlerësim): {sample_mb:.1f} MB")
print(f"Kufiri i GitHub: {GITHUB_LIMIT_MB} MB")


Madhësia e skedarëve
Dataseti origjinal: 22.0 MB
Në memorie (pandas): 31.9 MB


Versioni i pastruar (vlerësim): 17.5 MB
Mostra 10% (vlerësim): 2.3 MB
Kufiri i GitHub: 100 MB


## 2. DEFINIMI I TIPEVE TË TË DHËNAVE

In [4]:
section("2. DEFINIMI I TIPEVE TË TË DHËNAVE")

# Për secilën kolonë: tipi konceptual dhe tipi i synuar në pandas (shih reports/data_types.md)
TYPES = pd.DataFrame([
    ("gbifID",                        "ID",       "int64"),
    ("scientificName",                "nominal",  "category"),
    ("species",                       "nominal",  "category"),
    ("vernacularName",                "nominal",  "category"),
    ("kingdom",                       "nominal",  "category"),
    ("phylum",                        "nominal",  "category"),
    ("class",                         "nominal",  "category"),
    ("order",                         "nominal",  "category"),
    ("family",                        "nominal",  "category"),
    ("genus",                         "nominal",  "category"),
    ("taxonRank",                     "ordinal",  "category (ordered)"),
    ("taxonKey",                      "ID",       "Int64"),
    ("speciesKey",                    "ID",       "Int64"),
    ("iucnRedListCategory",           "ordinal",  "category (ordered)"),
    ("iucnRedListStatus",             "ordinal",  "category (ordered)"),
    ("eventDate",                     "datë",     "datetime64"),
    ("year",                          "interval", "Int64"),
    ("month",                         "ordinal",  "Int64"),
    ("day",                           "ordinal",  "Int64"),
    ("decimalLatitude",               "interval", "float64"),
    ("decimalLongitude",              "interval", "float64"),
    ("coordinateUncertaintyInMeters", "ratio",    "float64"),
    ("elevation",                     "interval", "float64"),
    ("locality",                      "tekst",    "str"),
    ("municipality",                  "nominal",  "category"),
    ("district",                      "nominal",  "category"),
    ("protectedArea",                 "nominal",  "category"),
    ("protectedAreaDesignation",      "nominal",  "category"),
    ("strictlyProtected",             "binar",    "bool"),
    ("basisOfRecord",                 "nominal",  "category"),
    ("individualCount",               "ratio",    "Int64"),
    ("establishmentMeans",            "nominal",  "category"),
    ("recordedBy",                    "tekst",    "str"),
    ("identifiedBy",                  "tekst",    "str"),
    ("institutionCode",               "nominal",  "category"),
    ("collectionCode",                "nominal",  "category"),
    ("catalogNumber",                 "ID",       "str"),
    ("datasetKey",                    "ID",       "category"),
    ("license",                       "nominal",  "category"),
    ("issue",                         "tekst",    "str"),
    ("iasUnionConcern",               "binar",    "bool"),
    ("iasAppliesFrom",                "datë",     "datetime64"),
], columns=["kolona", "tipi_konceptual", "tipi_i_synuar"]).set_index("kolona")
assert list(TYPES.index) == list(df_raw.columns)

# Kolonat ordinale: renditja e kategorive dhe vlerat që nënkuptojnë "e panjohur" (-> NaN)
ORDERED = {
    "taxonRank": (["KINGDOM", "PHYLUM", "CLASS", "ORDER", "FAMILY", "GENUS",
                   "SPECIES", "SUBSPECIES", "VARIETY", "FORM"], ["UNRANKED"]),
    "iucnRedListCategory": (["LC", "NT", "VU", "EN", "CR", "EW", "EX"], ["NE", "DD"]),
    "iucnRedListStatus": (["Least Concern", "Near Threatened", "Vulnerable", "Endangered",
                           "Critically Endangered", "Extinct in the Wild", "Extinct"],
                          ["Not Evaluated", "Data Deficient"]),
}

df = df_raw.copy()
for col, target in TYPES.tipi_i_synuar.items():
    if col in ORDERED:
        cats, unknown = ORDERED[col]
        df[col] = pd.Categorical(df[col].replace({u: np.nan for u in unknown}),
                                 categories=cats, ordered=True)
    elif target == "datetime64":
        # te intervalet "fillimi/fundi" merret data e fillimit; ora hiqet
        start = df[col].str.split("/").str[0].str[:10]
        df[col] = pd.to_datetime(start, format="ISO8601", errors="coerce")
    elif target == "str":
        pass  # teksti mbetet ashtu siç lexohet (str në pandas 3, object në pandas 2)
    elif target != str(df[col].dtype):
        df[col] = df[col].astype(target)


def dtype_label(s):
    if isinstance(s.dtype, pd.CategoricalDtype):
        return "category (ordered)" if s.cat.ordered else "category"
    if s.dtype.kind == "M":
        return "datetime64"
    return "str" if s.dtype.kind in "OUT" or isinstance(s.dtype, pd.StringDtype) else str(s.dtype)


type_report = TYPES.assign(
    tipi_pandas=df_raw.dtypes.astype(str),
    tipi_pas=[dtype_label(df[c]) for c in TYPES.index],
    vlera_unike=df_raw.nunique(),
    na_para=df_raw.isna().sum(),
    na_pas=df.isna().sum(),
)[["tipi_pandas", "tipi_konceptual", "tipi_i_synuar", "tipi_pas", "vlera_unike", "na_para", "na_pas"]]
assert (type_report.tipi_i_synuar == type_report.tipi_pas).all()

print(type_report.to_string())
print("\nKolona sipas tipit konceptual:")
print(type_report.tipi_konceptual.value_counts().to_string())

mem_before = df_raw.memory_usage(deep=True).sum() / 1024**2
mem_after = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nMemoria: {mem_before:.1f} MB -> {mem_after:.1f} MB")

# Vlerat që u bënë NaN gjatë konvertimit
new_na = type_report[type_report.na_pas != type_report.na_para]
print("\nVlera që u kthyen në NaN gjatë konvertimit:")
print((new_na.na_pas - new_na.na_para).rename("NaN_te_reja").to_string())


2. DEFINIMI I TIPEVE TË TË DHËNAVE


                              tipi_pandas tipi_konceptual       tipi_i_synuar            tipi_pas  vlera_unike  na_para  na_pas
kolona                                                                                                                         
gbifID                              int64              ID               int64               int64        46831        0       0
scientificName                        str         nominal            category            category         4647       37      37
species                               str         nominal            category            category         4095     1213    1213
vernacularName                        str         nominal            category            category         3314     3873    3873
kingdom                               str         nominal            category            category            7       37      37
phylum                                str         nominal            category            category       

## 3.1 VLERAT E ZBRAZËTA

In [5]:
section("3.1 VLERAT E ZBRAZËTA")
missing = pd.DataFrame({
    "mungojne": df_raw.isna().sum(),
    "perqindja": (df_raw.isna().mean() * 100).round(2),
}).sort_values("perqindja", ascending=False)
print(missing[missing.mungojne > 0])

print("\nRreshta pa asnjë vlerë që mungon:", df_raw.dropna().shape[0])
print("Kolona komplet:", missing.index[missing.mungojne == 0].tolist())

print(df_raw["iucnRedListCategory"].value_counts(dropna=False))


3.1 VLERAT E ZBRAZËTA
                               mungojne  perqindja
iasAppliesFrom                    46807      99.95
establishmentMeans                46591      99.49
identifiedBy                      39096      83.48
elevation                         37011      79.03
coordinateUncertaintyInMeters     32004      68.34
protectedArea                     25762      55.01
protectedAreaDesignation          25762      55.01
individualCount                   17697      37.79
locality                          15251      32.57
collectionCode                    13469      28.76
catalogNumber                     12559      26.82
vernacularName                     3873       8.27
recordedBy                         3853       8.23
institutionCode                    3711       7.92
day                                3219       6.87
month                              2867       6.12
year                               2268       4.84
eventDate                          2190       4.68
issue   

## 3.2 DUPLIKATET

In [6]:
section("3.2 DUPLIKATET")
print("Rreshta identikë:", df_raw.duplicated().sum())
print("gbifID të përsëritura:", df_raw["gbifID"].duplicated().sum())

key = ["scientificName", "decimalLatitude", "decimalLongitude", "eventDate"]
print("Vëzhgime të mundshme të dyfishta (takson + koordinata + data):", df_raw.duplicated(subset=key).sum())


3.2 DUPLIKATET
Rreshta identikë: 0
gbifID të përsëritura: 0


Vëzhgime të mundshme të dyfishta (takson + koordinata + data): 3336


## 3.3 FORMATET E eventDate

In [7]:
section("3.3 FORMATET E eventDate")
dates = df_raw["eventDate"].dropna()
fmt = (dates.str.replace(r"\d", "9", regex=True)
            .value_counts()
            .rename_axis("formati")
            .to_frame("regjistrime"))
print(fmt)

start = df_raw["eventDate"].str.split("/").str[0].str[:10]
# eventDate është konvertuar te seksioni 2 (data e fillimit)
df["eventDatePrecision"] = pd.Categorical(
    start.str.len().map({4: "year", 7: "month", 10: "day"})
)
df["isDateRange"] = df_raw["eventDate"].str.contains("/", na=False)

print("\nData të palexueshme:", (df["eventDate"].isna() & df_raw["eventDate"].notna()).sum())
print("Intervale datash:", df["isDateRange"].sum())
print(df["eventDatePrecision"].value_counts(dropna=False))


both = df["eventDate"].notna() & df["year"].notna()
mismatch = both & (df["eventDate"].dt.year != df["year"])
print("\nMospërputhje eventDate vs year:", mismatch.sum())
print("Data në të ardhmen:", (df["eventDate"] > pd.Timestamp.today()).sum())


3.3 FORMATET E eventDate


                                     regjistrime
formati                                         
9999-99-99                                 37945
9999-99-99T99:99:99                         2554
9999-99-99T99:99                            2417
9999-99-99T99:99:99.999Z                     595
9999                                         409
9999-99-99/9999-99-99                        398
9999-99                                      173
9999-99-99T99:99:99Z                          67
9999/9999                                     46
9999-99-99T99:99Z                             34
9999-99-99T99:99Z/9999-99-99T99:99Z            3

Data të palexueshme: 0
Intervale datash: 447
eventDatePrecision
day      44013
NaN       2190
year       455
month      173
Name: count, dtype: int64

Mospërputhje eventDate vs year: 0
Data në të ardhmen: 0


## 3.4 VLERAT E DYSHIMTA

In [8]:
section("3.4 VLERAT E DYSHIMTA")
print(df[["year", "decimalLatitude", "decimalLongitude", "coordinateUncertaintyInMeters",
          "elevation", "individualCount"]].describe().T)

LAT, LON = (41.85, 43.27), (20.01, 21.80)
ELEV = (297, 2656)

checks = {
    "Koordinata jashtë Kosovës": ~df.decimalLatitude.between(*LAT) | ~df.decimalLongitude.between(*LON),
    "Lartësia = 0 m": df.elevation == 0,
    "Lartësia jashtë 297–2656 m (pa 0)": df.elevation.notna() & (df.elevation != 0) & ~df.elevation.between(*ELEV),
    "Pasiguria e koordinatave > 10 km": df.coordinateUncertaintyInMeters > 10_000,
    "Viti para 1900": df.year < 1900,
    "Identifikim mbi nivelin e species": df.species.isna(),
}
quality = pd.DataFrame({
    "regjistrime": {k: int(v.sum()) for k, v in checks.items()},
})
quality["perqindja"] = (quality.regjistrime / len(df) * 100).round(2)
print()
print(quality)


ic = df["individualCount"].dropna()
q1, q3 = ic.quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print(f"\nQ1={q1}, Q3={q3}, kufiri i sipërm IQR={upper}")
print("Outliers:", (ic > upper).sum(), "| Maksimumi:", ic.max())
print(ic.value_counts().head(10))


3.4 VLERAT E DYSHIMTA
                                 count         mean           std        min        25%        50%        75%        max
year                           44563.0  2015.648161     14.666617     1826.0     2018.0     2020.0     2022.0     2026.0
decimalLatitude                46831.0    42.564153      0.214627  41.858976  42.561417  42.579243  42.659067   43.24167
decimalLongitude               46831.0    20.972567      0.345911  20.020786  20.815317  21.048304  21.144554  21.762903
coordinateUncertaintyInMeters  14827.0  4865.172658  37074.786607        1.0      250.0     7071.0     7071.0  4226661.0
elevation                       9820.0   582.211845    634.895781        0.0        0.0      490.0      802.0     3050.0
individualCount                29134.0    22.948273    322.478196        1.0        1.0        2.0        6.0    10200.0

                                   regjistrime  perqindja
Koordinata jashtë Kosovës                    0       0.00
Lartësia = 0 

## 3.5 FLAMUJT E GBIF (issue)

In [9]:
section("3.5 FLAMUJT E GBIF (issue)")
issues = (df_raw["issue"].dropna()
          .str.split(";")
          .explode()
          .value_counts()
          .rename_axis("flamuri")
          .to_frame("regjistrime"))
issues["perqindja"] = (issues.regjistrime / len(df_raw) * 100).round(2)
print(issues.head(15))


3.5 FLAMUJT E GBIF (issue)


                                                  regjistrime  perqindja
flamuri                                                                 
CONTINENT_DERIVED_FROM_COORDINATES                      44725      95.50
TAXON_CONCEPT_ID_NOT_FOUND                              26628      56.86
COORDINATE_ROUNDED                                      14944      31.91
COUNTRY_DERIVED_FROM_COORDINATES                         7505      16.03
TAXON_ID_NOT_FOUND                                       6083      12.99
GEODETIC_DATUM_ASSUMED_WGS84                             3850       8.22
OCCURRENCE_STATUS_INFERRED_FROM_INDIVIDUAL_COUNT         1927       4.11
INSTITUTION_MATCH_NONE                                   1351       2.88
TAXON_MATCH_HIGHERRANK                                    756       1.61
RECORDED_DATE_INVALID                                     508       1.08
INSTITUTION_MATCH_FUZZY                                   452       0.97
COUNTRY_INVALID                                    

## 3.6 KONSISTENCA E VLERAVE KATEGORIKE

In [10]:
section("3.6 KONSISTENCA E VLERAVE KATEGORIKE")
for c in ["kingdom", "taxonRank", "basisOfRecord", "district", "license", "strictlyProtected", "iasUnionConcern"]:
    print(f"--- {c} ---")
    print(df[c].value_counts(dropna=False).to_string(), "\n")


3.6 KONSISTENCA E VLERAVE KATEGORIKE
--- kingdom ---
kingdom
Animalia          42003
Plantae            3740
Fungi              1006
Chromista            38
NaN                  37
Protozoa              4
incertae sedis        2
Bacteria              1 

--- taxonRank ---
taxonRank
SPECIES       44727
GENUS           925
SUBSPECIES      579
NaN             251
FORM            132
VARIETY          60
FAMILY           58
ORDER            41
KINGDOM          27
PHYLUM           17
CLASS            14 

--- basisOfRecord ---
basisOfRecord
HUMAN_OBSERVATION      42479
PRESERVED_SPECIMEN      1748
OCCURRENCE              1269
MATERIAL_SAMPLE          641
MACHINE_OBSERVATION      376
MATERIAL_CITATION        317
OBSERVATION                1 

--- district ---
district
Prishtinë / Priština              25734
Prizren                            5268
Gjilan / Gnjilane                  4156
Mitrovicë / Kosovska Mitrovica     3196
Pejë / Peć                         3187
Gjakovë / Đakovica         

## 3.7 KORNIZA E KUALITETIT

Pesë dimensione: plotësia, vlefshmëria, konsistenca, unikaliteti dhe aktualiteti. Për secilin kontroll: metrika, rregulli dhe kolonat. Përshkrimi i plotë është te `reports/quality_framework.md`.

In [11]:
section("3.7 KORNIZA E KUALITETIT")
# Çdo kontroll ka: dimensionin, kolonat, rreshtat ku aplikohet dhe rreshtat që nuk e plotësojnë.
# Metrika = rreshta që e plotësojnë / rreshta ku aplikohet (shih reports/quality_framework.md).

TODAY = pd.Timestamp.today().normalize()
KEY_COLS = ["species", "eventDate", "decimalLatitude", "decimalLongitude",
            "municipality", "district", "basisOfRecord", "kingdom"]

VOCAB = {
    "kingdom": {"Animalia", "Plantae", "Fungi", "Chromista", "Protozoa", "Bacteria",
                "Archaea", "Viruses", "incertae sedis"},
    "taxonRank": {"KINGDOM", "PHYLUM", "CLASS", "ORDER", "FAMILY", "GENUS", "SPECIES",
                  "SUBSPECIES", "VARIETY", "FORM", "UNRANKED"},
    "basisOfRecord": {"HUMAN_OBSERVATION", "MACHINE_OBSERVATION", "OBSERVATION", "PRESERVED_SPECIMEN",
                      "FOSSIL_SPECIMEN", "LIVING_SPECIMEN", "MATERIAL_SAMPLE", "MATERIAL_CITATION",
                      "OCCURRENCE"},
    "iucnRedListCategory": {"LC", "NT", "VU", "EN", "CR", "EW", "EX", "NE", "DD"},
    "license": {"CC0_1_0", "CC_BY_4_0", "CC_BY_NC_4_0"},
    "establishmentMeans": {"native", "nativeReintroduced", "introduced",
                           "introducedAssistedColonisation", "vagrant", "uncertain"},
}
# Flamujt e GBIF që tregojnë vlerë të pavlefshme (jo vetëm interpretim informues)
GBIF_INVALID = ["TAXON_MATCH_NONE", "TAXON_MATCH_FUZZY", "TAXON_MATCH_HIGHERRANK",
                "RECORDED_DATE_INVALID", "INDIVIDUAL_COUNT_INVALID", "BASIS_OF_RECORD_INVALID",
                "COORDINATE_PRECISION_INVALID", "COORDINATE_UNCERTAINTY_METERS_INVALID"]

ALL = pd.Series(True, index=df.index)
rules = []


def add_rule(id_, dim, kontrolli, kolonat, njesia, n, n_fail, veprimi):
    rules.append({"id": id_, "dimensioni": dim, "kontrolli": kontrolli, "kolonat": kolonat,
                  "njesia": njesia, "aplikohen": n, "nuk_plotesojne": n_fail,
                  "perqindja": round((1 - n_fail / n) * 100, 2) if n else np.nan,
                  "veprimi": veprimi})


def rule(id_, dim, kontrolli, kolonat, fail, applies=ALL, veprimi=""):
    fail = pd.Series(fail, index=df.index).fillna(False).astype(bool) & applies
    add_rule(id_, dim, kontrolli, kolonat, "rreshta", int(applies.sum()), int(fail.sum()), veprimi)
    return fail


def many_to_one(child, parent, within=()):
    """Rreshtat ku vlera e `child` (brenda grupit `within`) lidhet me më shumë se një `parent`."""
    keys = [*within, child]
    n_parents = df_raw.groupby(keys)[parent].transform("nunique")
    return (n_parents > 1) & df_raw[keys + [parent]].notna().all(axis=1)


# --- Plotësia ---
D = "Plotësia"
rule("P1", D, "Të gjitha kolonat kryesore të plota", ", ".join(KEY_COLS),
     df[KEY_COLS].isna().any(axis=1), veprimi="analiza kryesore vetëm me rreshtat e plotë")
rule("P2", D, "Identifikim deri në species", "species", df.species.isna(),
     veprimi="përjashto nga analiza sipas species")
rule("P3", D, "Data e njohur", "eventDate", df.eventDate.isna(),
     veprimi="përjashto nga analiza kohore")
rule("P4", D, "Statusi IUCN i vlerësuar (NE/DD = mungesë)", "iucnRedListCategory",
     df.iucnRedListCategory.isna(), veprimi="shëno 'i pavlerësuar'")
rule("P5", D, "Lartësia e njohur (0 m = mungesë)", "elevation",
     df.elevation.isna() | (df.elevation == 0), veprimi="0 -> NaN; përdor vetëm si veti ndihmëse")
rule("P6", D, "Pasiguria e koordinatave e dhënë", "coordinateUncertaintyInMeters",
     df.coordinateUncertaintyInMeters.isna(), veprimi="shëno 'e panjohur' te diskretizimi")
rule("P7", D, "Numri i individëve i dhënë", "individualCount", df.individualCount.isna(),
     veprimi="mos e imputo; analizo vetëm vlerat e dhëna")
sparse = df_raw.columns[df_raw.isna().mean() > 0.8]
add_rule("P8", D, "Kolona me jo më shumë se 80% mungesa", ", ".join(sparse), "kolona",
         df_raw.shape[1], len(sparse), "hiq kolonat")

# --- Vlefshmëria ---
D = "Vlefshmëria"
LAT, LON, ELEV = (41.85, 43.27), (20.01, 21.80), (297, 2656)
has_coord = df.decimalLatitude.notna() & df.decimalLongitude.notna()
rule("V1", D, f"Koordinatat brenda Kosovës (lat {LAT[0]}–{LAT[1]}, lon {LON[0]:.2f}–{LON[1]:.2f})", "decimalLatitude, decimalLongitude",
     ~df.decimalLatitude.between(*LAT) | ~df.decimalLongitude.between(*LON), has_coord, "largo rreshtin")
elev = df.elevation.notna() & (df.elevation != 0)
rule("V2", D, f"Lartësia brenda {ELEV[0]}–{ELEV[1]} m", "elevation",
     ~df.elevation.between(*ELEV), elev, "vendos NaN")
unc = df.coordinateUncertaintyInMeters.notna()
rule("V3", D, "Pasiguria > 0 dhe ≤ 10 km", "coordinateUncertaintyInMeters",
     (df.coordinateUncertaintyInMeters <= 0) | (df.coordinateUncertaintyInMeters > 10_000), unc,
     "shëno; përjashto nga analiza hapësinore e detajuar")
rule("V4", D, "eventDate e lexueshme si ISO 8601", "eventDate",
     df.eventDate.isna(), df_raw.eventDate.notna(), "vendos NaN")
rule("V5", D, "Data jo në të ardhmen", "eventDate, year",
     (df.eventDate > TODAY) | (df.year > TODAY.year), df.eventDate.notna() | df.year.notna(), "vendos NaN")
has_ymd = df[["year", "month", "day"]].notna().all(axis=1)
ymd = pd.to_datetime(df.loc[has_ymd, ["year", "month", "day"]].astype("int64"), errors="coerce")
rule("V6", D, "month 1–12, day 1–31 dhe data year-month-day ekziston", "year, month, day",
     ~df.month.between(1, 12) | ~df.day.between(1, 31) | ymd.reindex(df.index).isna(),
     has_ymd, "vendos NaN")
rule("V7", D, "individualCount ≥ 1", "individualCount", df.individualCount < 1,
     df.individualCount.notna(), "vendos NaN")
vocab_fail = pd.concat([df_raw[c].notna() & ~df_raw[c].isin(v) for c, v in VOCAB.items()], axis=1).any(axis=1)
rule("V8", D, "Vlerat kategorike brenda fjalorit të lejuar (GBIF/DwC/IUCN)", ", ".join(VOCAB),
     vocab_fail, veprimi="korrigjo ose vendos NaN")
flags = df_raw.issue.fillna("").str.split(";")
rule("V9", D, "Pa flamuj GBIF të pavlefshmërisë", "issue",
     flags.apply(lambda f: any(x in GBIF_INVALID for x in f)), veprimi="shëno me flamur binar")
ic = df.individualCount.dropna()
q1, q3 = ic.quantile([0.25, 0.75])
rule("V10", D, f"individualCount jo outlier (IQR, > {q3 + 1.5 * (q3 - q1):g})", "individualCount",
     df.individualCount > q3 + 1.5 * (q3 - q1), df.individualCount.notna(), "shëno; log1p te transformimi")

# --- Konsistenca ---
D = "Konsistenca"
start = df_raw.eventDate.str.split("/").str[0].str[:10]
prec = start.str.len()
ev = df.eventDate
date_fail = ((ev.dt.year != df.year)
             | ((prec >= 7) & (ev.dt.month != df.month))
             | ((prec == 10) & (ev.dt.day != df.day)))
rule("K1", D, "eventDate përputhet me year/month/day (sipas precizionit)", "eventDate, year, month, day",
     date_fail, ev.notna() & df.year.notna(), "përdor eventDate; plotëso year/month/day prej saj")
tax_pairs = [("genus", "family"), ("family", "order"), ("order", "class"), ("class", "phylum"),
             ("phylum", "kingdom")]
# brenda të njëjtit kingdom: p.sh. Linaria dhe Prunella janë gjini edhe te zogjtë edhe te bimët
rule("K2", D, "Hierarkia taksonomike: çdo takson ka një prind (brenda kingdom)",
     "genus, family, order, class, phylum, kingdom",
     pd.concat([many_to_one(c, p, within=["kingdom"]) for c, p in tax_pairs], axis=1).any(axis=1),
     df.kingdom.notna(), "rishiko taksonët me më shumë se një prind")
sp_gen = df_raw.species.notna() & df_raw.genus.notna()
rule("K3", D, "species fillon me genus", "species, genus",
     df_raw.species.str.split().str[0] != df_raw.genus, sp_gen, "rishiko")
infra = df_raw.taxonRank.isin(["SPECIES", "SUBSPECIES", "VARIETY", "FORM"])
rule("K4", D, "taxonRank SPECIES ose më i ulët ⇔ species e plotë", "taxonRank, species",
     infra != df_raw.species.notna(), df_raw.taxonRank.notna() & (df_raw.taxonRank != "UNRANKED"), "rishiko")
rule("K5", D, "iucnRedListCategory ↔ iucnRedListStatus 1:1", "iucnRedListCategory, iucnRedListStatus",
     many_to_one("iucnRedListCategory", "iucnRedListStatus") | many_to_one("iucnRedListStatus", "iucnRedListCategory"),
     df_raw.iucnRedListCategory.notna(), "mbaj iucnRedListCategory")
rule("K6", D, "Çdo komunë i përket një rajoni", "municipality, district",
     many_to_one("municipality", "district"), veprimi="korrigjo rajonin")
rule("K7", D, "protectedArea dhe protectedAreaDesignation bashkë; një zonë = një lloj",
     "protectedArea, protectedAreaDesignation",
     (df_raw.protectedArea.isna() != df_raw.protectedAreaDesignation.isna())
     | many_to_one("protectedArea", "protectedAreaDesignation"), veprimi="korrigjo llojin")
rule("K8", D, "iasAppliesFrom e plotë ⇒ iasUnionConcern = TRUE", "iasAppliesFrom, iasUnionConcern",
     ~df_raw.iasUnionConcern, df_raw.iasAppliesFrom.notna(), "rishiko")
nominal = TYPES.index[TYPES.tipi_konceptual == "nominal"]
fmt_fail = pd.Series(False, index=df.index)
for c in nominal:
    s = df_raw[c].dropna().astype(str)
    variants = s.drop_duplicates().groupby(s.drop_duplicates().str.strip().str.lower()).size()
    bad = s[(s != s.str.strip()) | s.str.contains("  ", regex=False)
            | s.str.strip().str.lower().isin(variants[variants > 1].index)]
    fmt_fail |= df_raw[c].isin(bad.unique())
rule("K9", D, "Kategoritë pa hapësira të tepërta dhe pa variante të shkronjave", ", ".join(nominal),
     fmt_fail, veprimi="strip + njësim i shkronjave")

# --- Unikaliteti ---
D = "Unikaliteti"
rule("U1", D, "gbifID unik", "gbifID", df_raw.gbifID.duplicated(keep=False), veprimi="largo dyfishin")
rule("U2", D, "Pa rreshta identikë", "të gjitha", df_raw.duplicated(keep="first"), veprimi="largo dyfishin")
rule("U3", D, "catalogNumber unik brenda dataset-it", "datasetKey, catalogNumber",
     df_raw.duplicated(["datasetKey", "catalogNumber"], keep=False), df_raw.catalogNumber.notna(),
     "rishiko (mund të jenë disa pjesë të një ekzemplari)")
DUP_KEY = ["scientificName", "decimalLatitude", "decimalLongitude", "eventDate"]
rule("U4", D, "Pa vëzhgime të dyfishta (takson + koordinata + data)", ", ".join(DUP_KEY),
     df_raw.duplicated(DUP_KEY, keep="first"), df_raw[DUP_KEY].notna().all(axis=1),
     "shëno; vendos te integrimi")
dup_groups = df_raw[df_raw.duplicated(DUP_KEY, keep=False) & df_raw[DUP_KEY].notna().all(axis=1)]
multi_src = dup_groups.groupby(DUP_KEY).datasetKey.transform("nunique") > 1
rule("U5", D, "Pa dyfishime ndërmjet burimeve të ndryshme (datasetKey)", ", ".join(DUP_KEY) + ", datasetKey",
     df_raw.index.isin(multi_src[multi_src].index) & df_raw.duplicated(DUP_KEY, keep="first"),
     df_raw[DUP_KEY].notna().all(axis=1), "mbaj një rresht për grup")

# --- Aktualiteti ---
D = "Aktualiteti"
RECENT = TODAY.year - 9
has_year = df.year.notna()
rule("A1", D, f"Regjistrim nga 10 vitet e fundit (year ≥ {RECENT})", "year", df.year < RECENT, has_year,
     "krahaso periudhat; mos i përziej pa shënim")
rule("A2", D, "Regjistrim nga viti 2000 e tutje", "year", df.year < 2000, has_year,
     "shëno si historik")
last_year = df.groupby("datasetKey", observed=True).year.max()
inactive = last_year[last_year < RECENT].index
rule("A3", D, f"Dataset-i ka regjistrime pas vitit {RECENT - 1}", "datasetKey, year",
     df.datasetKey.isin(inactive), has_year, "shëno burimin si joaktiv")

METRIKA = {
    "P1": "% e rreshtave me të 8 kolonat kryesore të plota",
    "P2": "% e rreshtave me species të plotë",
    "P3": "% e rreshtave me eventDate të plotë",
    "P4": "% e rreshtave me kategori IUCN të vlerësuar (LC–EX)",
    "P5": "% e rreshtave me elevation të plotë dhe ≠ 0",
    "P6": "% e rreshtave me coordinateUncertaintyInMeters të plotë",
    "P7": "% e rreshtave me individualCount të plotë",
    "P8": "% e kolonave me ≤ 80% mungesa",
    "V1": "% e koordinatave brenda kutisë gjeografike të Kosovës",
    "V2": "% e lartësive (≠ 0) brenda 297–2656 m",
    "V3": "% e pasigurive të dhëna në intervalin (0, 10 000] m",
    "V4": "% e eventDate të dhëna që lexohen si datë",
    "V5": "% e datave ≤ data e sotme",
    "V6": "% e treshave year-month-day që formojnë datë reale",
    "V7": "% e individualCount të dhëna ≥ 1",
    "V8": "% e rreshtave me të gjitha vlerat kategorike brenda fjalorit",
    "V9": "% e rreshtave pa asnjë flamur GBIF të pavlefshmërisë",
    "V10": "% e individualCount të dhëna ≤ Q3 + 1.5·IQR",
    "K1": "% e rreshtave ku viti/muaji/dita e eventDate = year/month/day",
    "K2": "% e rreshtave ku çdo takson ka një prind brenda kingdom",
    "K3": "% e rreshtave ku fjala e parë e species = genus",
    "K4": "% e rreshtave ku taxonRank dhe species përputhen",
    "K5": "% e rreshtave ku kodi dhe emri IUCN lidhen 1:1",
    "K6": "% e rreshtave ku komuna lidhet me një rajon",
    "K7": "% e rreshtave ku zona e mbrojtur dhe lloji i saj përputhen",
    "K8": "% e rreshtave me iasAppliesFrom ku iasUnionConcern = TRUE",
    "K9": "% e rreshtave pa hapësira të tepërta ose variante shkronjash",
    "U1": "% e gbifID unike",
    "U2": "% e rreshtave që nuk janë kopje e një rreshti tjetër",
    "U3": "% e catalogNumber unike brenda datasetKey",
    "U4": "% e vëzhgimeve pa dyfish (takson + koordinata + data)",
    "U5": "% e vëzhgimeve pa dyfish nga një dataset tjetër",
    "A1": "% e regjistrimeve me year ≥ 2017",
    "A2": "% e regjistrimeve me year ≥ 2000",
    "A3": "% e regjistrimeve nga dataset-e me të dhëna pas 2016",
}
framework = pd.DataFrame(rules).set_index("id")
framework.insert(1, "metrika", framework.index.map(METRIKA))
assert framework.metrika.notna().all()

print("Tabela: Dimensioni | Metrika | Kontrolli | Kolonat")
display(framework[["dimensioni", "metrika", "kontrolli", "kolonat"]]
        .rename(columns=str.capitalize).style.set_properties(**{"text-align": "left"}))
print()
print("Vlerat fillestare:")
print(framework[["dimensioni", "kontrolli", "njesia", "aplikohen", "nuk_plotesojne", "perqindja"]].to_string())
print("\nKontrolle sipas dimensionit (gjithsej / 100% të plotësuara):")
print(framework.groupby("dimensioni", sort=False).perqindja
      .agg(kontrolle="size", plot=lambda x: int((x == 100).sum())).to_string())
print(f"\nViti median: {int(df.year.median())}  |  Regjistrimi më i ri: {df.eventDate.max().date()}  |  "
      f"Dataset-e joaktive: {len(inactive)} / {df.datasetKey.nunique()}")


3.7 KORNIZA E KUALITETIT


Tabela: Dimensioni | Metrika | Kontrolli | Kolonat


,Dimensioni,Metrika,Kontrolli,Kolonat
id,,,,
P1,Plotësia,% e rreshtave me të 8 kolonat kryesore të plota,Të gjitha kolonat kryesore të plota,"species, eventDate, decimalLatitude, decimalLongitude, municipality, district, basisOfRecord, kingdom"
P2,Plotësia,% e rreshtave me species të plotë,Identifikim deri në species,species
P3,Plotësia,% e rreshtave me eventDate të plotë,Data e njohur,eventDate
P4,Plotësia,% e rreshtave me kategori IUCN të vlerësuar (LC–EX),Statusi IUCN i vlerësuar (NE/DD = mungesë),iucnRedListCategory
P5,Plotësia,% e rreshtave me elevation të plotë dhe ≠ 0,Lartësia e njohur (0 m = mungesë),elevation
P6,Plotësia,% e rreshtave me coordinateUncertaintyInMeters të plotë,Pasiguria e koordinatave e dhënë,coordinateUncertaintyInMeters
P7,Plotësia,% e rreshtave me individualCount të plotë,Numri i individëve i dhënë,individualCount
P8,Plotësia,% e kolonave me ≤ 80% mungesa,Kolona me jo më shumë se 80% mungesa,"establishmentMeans, identifiedBy, iasAppliesFrom"
V1,Vlefshmëria,% e koordinatave brenda kutisë gjeografike të Kosovës,"Koordinatat brenda Kosovës (lat 41.85–43.27, lon 20.01–21.80)","decimalLatitude, decimalLongitude"



Vlerat fillestare:
      dimensioni                                                               kontrolli   njesia  aplikohen  nuk_plotesojne  perqindja
id                                                                                                                                     
P1      Plotësia                                     Të gjitha kolonat kryesore të plota  rreshta      46831            3205      93.16
P2      Plotësia                                             Identifikim deri në species  rreshta      46831            1213      97.41
P3      Plotësia                                                           Data e njohur  rreshta      46831            2190      95.32
P4      Plotësia                              Statusi IUCN i vlerësuar (NE/DD = mungesë)  rreshta      46831           16935      63.84
P5      Plotësia                                       Lartësia e njohur (0 m = mungesë)  rreshta      46831           40645      13.21
P6      Plotësia            

## 3.8 PLOTËSIA PËR KOLONË DHE PËR RRESHT

Plotësia për kolonë llogaritet edhe me **mungesat e fshehura**, vlera që duken të plota por nënkuptojnë "e panjohur" (`elevation = 0`, `NE`/`DD`, `UNRANKED`, `unknown`). NaN te `protectedArea` (jashtë zonës) dhe te `iasAppliesFrom` (vlen vetëm për speciet pushtuese) nuk numërohen si mungesa.

Plotësia për rresht: sa rreshta i kanë të plota fushat kyçe **species, data, koordinata dhe komuna**. Përshkrimi është te `reports/completeness.md`.

In [12]:
section("3.8 PLOTËSIA PËR KOLONË DHE PËR RRESHT")
N = len(df_raw)


def norm(col):
    return df_raw[col].astype("string").str.strip().str.lower()


# Mungesat e fshehura: vlera që duken të plota, por nënkuptojnë "e panjohur"
HIDDEN = {
    "elevation": ("0 m", df_raw.elevation == 0),
    "iucnRedListCategory": ("NE, DD", df_raw.iucnRedListCategory.isin(["NE", "DD"])),
    "iucnRedListStatus": ("Not Evaluated, Data Deficient",
                          df_raw.iucnRedListStatus.isin(["Not Evaluated", "Data Deficient"])),
    "taxonRank": ("UNRANKED", df_raw.taxonRank == "UNRANKED"),
    "locality": ("unknown", norm("locality") == "unknown"),
    "recordedBy": ("unknown", norm("recordedBy") == "unknown"),
    "identifiedBy": ("0, Anon., unknown", norm("identifiedBy").isin(["0", "anon.", "unknown"])),
}
# Mungesat strukturore: NaN ka kuptim ose kolona nuk vlen për rreshtin
NAN_MEANS_VALUE = {
    "protectedArea": "NaN = jashtë zonës së mbrojtur",
    "protectedAreaDesignation": "NaN = jashtë zonës së mbrojtur",
}
APPLIES = {"iasAppliesFrom": ("vlen vetëm kur iasUnionConcern = TRUE", df_raw.iasUnionConcern)}

# Mungesa që mund të plotësohen nga eventDate (sipas precizionit të datës)
start = df_raw.eventDate.str.split("/").str[0].str[:10]
prec = start.str.len()
RECOVERABLE = {
    "year": df_raw.year.isna() & (prec >= 4),
    "month": df_raw.month.isna() & (prec >= 7),
    "day": df_raw.day.isna() & (prec == 10),
}

applies = pd.DataFrame(True, index=df_raw.index, columns=df_raw.columns)
missing_real = df_raw.isna()
for c, (_, mask) in APPLIES.items():
    applies[c] = mask
for c in NAN_MEANS_VALUE:
    missing_real[c] = False
for c, (_, mask) in HIDDEN.items():
    missing_real[c] |= mask.fillna(False)
missing_real &= applies

col_completeness = pd.DataFrame({
    "aplikohen": applies.sum(),
    "nan": df_raw.isna().sum(),
    "te_fshehura": pd.Series({c: int(m.fillna(False).sum()) for c, (_, m) in HIDDEN.items()}),
    "vlerat_e_fshehura": pd.Series({c: v for c, (v, _) in HIDDEN.items()}),
    "mungesa_reale": missing_real.sum(),
    "te_rikuperueshme": pd.Series({c: int(m.sum()) for c, m in RECOVERABLE.items()}),
}).reindex(df_raw.columns)
int_cols = ["aplikohen", "nan", "te_fshehura", "mungesa_reale", "te_rikuperueshme"]
col_completeness[int_cols] = col_completeness[int_cols].fillna(0).astype(int)
col_completeness["vlerat_e_fshehura"] = col_completeness.vlerat_e_fshehura.fillna("")
col_completeness["plotesia_e_dukshme"] = (df_raw.notna().mean() * 100).round(2)
col_completeness["plotesia_reale"] = ((1 - col_completeness.mungesa_reale / col_completeness.aplikohen)
                                      * 100).round(2)
col_completeness["shenim"] = pd.Series({**NAN_MEANS_VALUE, **{c: v for c, (v, _) in APPLIES.items()},
                                        **{c: "plotësohet nga eventDate" for c, m in RECOVERABLE.items()
                                           if m.any()}}).reindex(df_raw.columns).fillna("")
col_completeness = col_completeness.sort_values(["plotesia_reale", "plotesia_e_dukshme"])
col_completeness.index.name = "kolona"

print("Tabela e plotësisë për kolonë (renditur nga më e ulëta)")
display(col_completeness)
diff = col_completeness[col_completeness.plotesia_reale != col_completeness.plotesia_e_dukshme]
print(f"Kolona ku plotësia reale ndryshon nga ajo e dukshme: {len(diff)}")
print(f"Kolona 100% të plota: {(col_completeness.plotesia_reale == 100).sum()} / {df_raw.shape[1]}")

# --- Plotësia për rresht: fushat kyçe ---
KEY_FIELDS = {
    "species": df.species.notna(),
    "data (eventDate)": df.eventDate.notna(),
    "koordinata (lat + lon)": df.decimalLatitude.notna() & df.decimalLongitude.notna(),
    "komuna (municipality)": df.municipality.notna(),
}
key = pd.DataFrame(KEY_FIELDS)
n_missing = (~key).sum(axis=1)


def share(counts, tabela):
    out = counts.rename("rreshta").to_frame()
    out["perqindja"] = (out.rreshta / N * 100).round(2)
    out.insert(0, "tabela", tabela)
    out.index.name = "kategoria"
    return out


per_field = share(key.sum().astype(int), "fusha kyçe e plotë")
all_key = share(pd.Series({"të 4 fushat kyçe të plota": int((n_missing == 0).sum()),
                           "të paktën një fushë kyçe mungon": int((n_missing > 0).sum())}), "rreshta të plotë")
by_count = share(n_missing.value_counts().sort_index().rename(lambda k: "1 fushë kyçe mungon" if k == 1 else f"{k} fusha kyçe mungojnë"),
                 "numri i fushave që mungojnë")
pattern = (~key).apply(lambda r: " + ".join(k.split(" ")[0] for k, v in r.items() if v), axis=1)
by_pattern = share(pattern[n_missing > 0].value_counts().rename(lambda p: f"mungon: {p}"),
                   "çfarë mungon")

row_share = (1 - missing_real.sum(axis=1) / applies.sum(axis=1)) * 100
bins = pd.cut(row_share, [0, 50, 75, 90, 100], right=False, include_lowest=True,
              labels=["< 50%", "50–75%", "75–90%", "90–100%"])
bins = bins.cat.add_categories("100%")
bins[row_share == 100] = "100%"
by_bins = share(bins.value_counts().reindex(["< 50%", "50–75%", "75–90%", "90–100%", "100%"]),
                "plotësia e të 42 kolonave")

row_completeness = pd.concat([all_key, per_field, by_count, by_pattern, by_bins])
print("\nTabela e plotësisë për rresht")
display(row_completeness)
print(f"Plotësia mesatare e një rreshti (42 kolona, me mungesat e fshehura): {row_share.mean():.2f}%  |  "
      f"mediana: {row_share.median():.2f}%")


3.8 PLOTËSIA PËR KOLONË DHE PËR RRESHT


Tabela e plotësisë për kolonë (renditur nga më e ulëta)


,aplikohen,nan,te_fshehura,vlerat_e_fshehura,mungesa_reale,te_rikuperueshme,plotesia_e_dukshme,plotesia_reale,shenim
kolona,,,,,,,,,
establishmentMeans,46831,46591,0,,46591,0,0.51,0.51,
elevation,46831,37011,3634,0 m,40645,0,20.97,13.21,
identifiedBy,46831,39096,15,"0, Anon., unknown",39111,0,16.52,16.48,
coordinateUncertaintyInMeters,46831,32004,0,,32004,0,31.66,31.66,
individualCount,46831,17697,0,,17697,0,62.21,62.21,
iucnRedListCategory,46831,1213,15722,"NE, DD",16935,0,97.41,63.84,
iucnRedListStatus,46831,1213,15722,"Not Evaluated, Data Deficient",16935,0,97.41,63.84,
locality,46831,15251,38,unknown,15289,0,67.43,67.35,
collectionCode,46831,13469,0,,13469,0,71.24,71.24,


Kolona ku plotësia reale ndryshon nga ajo e dukshme: 9
Kolona 100% të plota: 13 / 42



Tabela e plotësisë për rresht


,tabela,rreshta,perqindja
kategoria,,,
të 4 fushat kyçe të plota,rreshta të plotë,43626,93.16
të paktën një fushë kyçe mungon,rreshta të plotë,3205,6.84
species,fusha kyçe e plotë,45618,97.41
data (eventDate),fusha kyçe e plotë,44641,95.32
koordinata (lat + lon),fusha kyçe e plotë,46831,100.00
komuna (municipality),fusha kyçe e plotë,46831,100.00
0 fusha kyçe mungojnë,numri i fushave që mungojnë,43626,93.16
1 fushë kyçe mungon,numri i fushave që mungojnë,3007,6.42
2 fusha kyçe mungojnë,numri i fushave që mungojnë,198,0.42


Plotësia mesatare e një rreshti (42 kolona, me mungesat e fshehura): 85.49%  |  mediana: 90.24%


## 3.9 VLEFSHMËRIA

Kontrollet: koordinatat brenda Kosovës, diapazoni i viteve, `month` 1–12 dhe `day` 1–31, formatet e `eventDate`, frekuenca e vlerës 7071 te `coordinateUncertaintyInMeters` dhe diapazoni i `elevation`. Kolona `korniza` lidh çdo kontroll me ID-në te `reports/quality_framework.md`. Përshkrimi është te `reports/validity.md`.

In [13]:
section("3.9 VLEFSHMËRIA")
publisher = df_raw.datasetKey.map(
    pd.read_csv(ROOT / "processed" / "gbif_datasets_cache.csv").set_index("datasetKey").publishingOrg)
validity = []


def check(id_, korniza, kontrolli, kolonat, invalid, applies, veprimi):
    invalid = pd.Series(invalid, index=df.index).fillna(False).astype(bool) & applies
    n = int(applies.sum())
    validity.append({"id": id_, "korniza": korniza, "kontrolli": kontrolli, "kolonat": kolonat,
                     "aplikohen": n, "te_pavlefshme": int(invalid.sum()),
                     "perqindja_e_vlefshme": round((1 - invalid.sum() / n) * 100, 2), "veprimi": veprimi})
    return invalid


# --- 1. Koordinatat brenda Kosovës ---
COORD_FLAGS = ["COUNTRY_COORDINATE_MISMATCH", "ZERO_COORDINATE", "COORDINATE_OUT_OF_RANGE", "COORDINATE_INVALID",
               "PRESUMED_SWAPPED_COORDINATE", "PRESUMED_NEGATED_LATITUDE", "PRESUMED_NEGATED_LONGITUDE"]
flags = df_raw.issue.fillna("").str.split(";")
decimals = df_raw.decimalLatitude.astype(str).str.split(".").str[1].str.len()
check("VL1", "V1", f"Koordinatat brenda kutisë së Kosovës (lat {LAT[0]}–{LAT[1]}, lon {LON[0]:.2f}–{LON[1]:.2f})",
      "decimalLatitude, decimalLongitude",
      ~df.decimalLatitude.between(*LAT) | ~df.decimalLongitude.between(*LON), has_coord, "largo rreshtin")
check("VL2", "V1", "Pika ka komunë të Kosovës", "municipality, decimalLatitude, decimalLongitude",
      df.municipality.isna(), has_coord, "largo rreshtin")
check("VL3", "V1", "Pa flamuj GBIF për koordinata të gabuara", "issue",
      flags.apply(lambda f: any(x in COORD_FLAGS for x in f)), ALL, "largo rreshtin")
check("VL4", "–", "Koordinata me të paktën 3 shifra dhjetore (≤ ~110 m)", "decimalLatitude",
      decimals < 3, has_coord, "shëno; jo për analizë në nivel fshati")

# --- 2. Diapazoni i viteve ---
check("VL5", "V5", f"Viti brenda 1758–{TODAY.year}", "year",
      ~df.year.between(1758, TODAY.year), df.year.notna(), "vendos NaN")
check("VL6", "A2", "Viti ≥ 1900 (jo regjistrim historik)", "year", df.year < 1900, df.year.notna(),
      "shëno si historik")

# --- 3. month 1–12 dhe day 1–31 ---
check("VL7", "V6", "month 1–12", "month", ~df.month.between(1, 12), df.month.notna(), "vendos NaN")
check("VL8", "V6", "day 1–31", "day", ~df.day.between(1, 31), df.day.notna(), "vendos NaN")
check("VL9", "V6", "year-month-day formon datë reale (p.sh. jo 30 shkurt)", "year, month, day",
      ymd.reindex(df.index).isna(), has_ymd, "vendos NaN")

# --- 4. Formatet e eventDate ---
raw_date = df_raw.eventDate
pattern = raw_date.str.replace(r"\d", "9", regex=True)
is_range = raw_date.str.contains("/", na=False)
end = pd.to_datetime(raw_date.where(is_range).str.split("/").str[1].str[:10], format="ISO8601", errors="coerce")
range_days = (end - df.eventDate).dt.days
check("VL10", "V4", "eventDate lexohet si ISO 8601", "eventDate", df.eventDate.isna(), raw_date.notna(),
      "vendos NaN")
check("VL11", "–", "Te intervalet: data e fundit ≥ data e fillimit", "eventDate", range_days < 0, is_range,
      "vendos NaN")
check("VL12", "–", "Te intervalet: gjatësia ≤ 1 vit", "eventDate", range_days > 365, is_range,
      "shëno; jo për analizë sezonale")
check("VL13", "V9", "Pa flamurin GBIF RECORDED_DATE_INVALID", "issue, eventDate",
      flags.apply(lambda f: "RECORDED_DATE_INVALID" in f), ALL, "data mbetet NaN")

# --- 5. Vlera 7071 te coordinateUncertaintyInMeters ---
check("VL14", "V3", "coordinateUncertaintyInMeters > 0 dhe ≤ 10 km", "coordinateUncertaintyInMeters",
      (df.coordinateUncertaintyInMeters <= 0) | (df.coordinateUncertaintyInMeters > 10_000), unc,
      "shëno; jo për analizë hapësinore të detajuar")
GRID_10KM = 7071  # 5000·√2: gjysma e diagonales së një qelize 10 × 10 km
check("VL15", "–", f"coordinateUncertaintyInMeters ≠ {GRID_10KM} (pikë, jo qendër e qelizës 10 km)",
      "coordinateUncertaintyInMeters", df.coordinateUncertaintyInMeters == GRID_10KM, unc,
      "shëno; komuna mund të jetë e pasaktë")

# --- 6. Diapazoni i elevation ---
check("VL16", "P5", "elevation ≠ 0 (0 = e panjohur)", "elevation", df.elevation == 0, df.elevation.notna(),
      "vendos NaN")
check("VL17", "V2", f"elevation brenda {ELEV[0]}–{ELEV[1]} m", "elevation", ~df.elevation.between(*ELEV), elev,
      "vendos NaN")

validity = pd.DataFrame(validity).set_index("id")
validity["perqindja_qe_deshtojne"] = (validity.te_pavlefshme / validity.aplikohen * 100).round(2)

# Tabela e kërkuar: Rregulla | Rreshta që dështojnë | % (e rreshtave ku rregulla aplikohet)
rule_table = pd.DataFrame({
    "Rregulla": validity.index + " – " + validity.kontrolli,
    "Rreshta që dështojnë": validity.te_pavlefshme,
    "%": validity.perqindja_qe_deshtojne,
}).reset_index(drop=True)
print("Tabela: Rregulla | Rreshta që dështojnë | %")
display(rule_table.style.hide(axis="index").set_properties(subset=["Rregulla"], **{"text-align": "left"})
        .format({"Rreshta që dështojnë": "{:,}", "%": "{:.2f}"}))
print("% = rreshta që dështojnë / rreshta ku rregulla aplikohet (ku vlera ekziston) × 100")
print("\nDetajet e kontrolleve")
display(validity)

# --- Tabelat me detaje ---
t_coord = pd.Series({
    "kutia e Kosovës": f"lat {df.decimalLatitude.min():.4f}–{df.decimalLatitude.max():.4f}, "
                       f"lon {df.decimalLongitude.min():.4f}–{df.decimalLongitude.max():.4f}",
    "koordinata të ndryshme": f"{df[['decimalLatitude', 'decimalLongitude']].drop_duplicates().shape[0]:,}",
    "koordinata me 1–2 shifra dhjetore": f"{int((decimals < 3).sum()):,}",
}, name="vlera")
top_points = (df_raw.groupby(["decimalLatitude", "decimalLongitude"])
              .agg(regjistrime=("gbifID", "size"), locality=("locality", "first"),
                   publikuesi=("datasetKey", lambda s: publisher[s.index].mode().iat[0]))
              .sort_values("regjistrime", ascending=False).head(5))
print("\nKoordinatat")
display(t_coord.to_frame())
print("Pikat më të përsëritura (hotspot-e të vëzhgimit, jo qendra të vendit):")
display(top_points)

year_bins = pd.cut(df.year.astype("float"), [1758, 1900, 1950, 2000, 2010, 2020, TODAY.year + 1], right=False,
                   labels=["1758–1899", "1900–1949", "1950–1999", "2000–2009", "2010–2019", f"2020–{TODAY.year}"])
t_years = year_bins.value_counts(sort=False).rename("regjistrime").to_frame()
t_years["perqindja"] = (t_years.regjistrime / df.year.notna().sum() * 100).round(2)
print(f"\nViti: min {int(df.year.min())}, max {int(df.year.max())}, median {int(df.year.median())}")
display(t_years)

t_formats = (pd.DataFrame({"formati": pattern, "interval": is_range, "lexohet": df.eventDate.notna()})
             .dropna(subset=["formati"])
             .groupby("formati")
             .agg(regjistrime=("lexohet", "size"), interval=("interval", "first"), te_lexueshme=("lexohet", "sum"))
             .sort_values("regjistrime", ascending=False))
t_formats["precizioni"] = t_formats.index.str.split("/").str[0].str[:10].str.len().map(
    {4: "vit", 7: "muaj", 10: "ditë"})
t_formats["perqindja"] = (t_formats.regjistrime / raw_date.notna().sum() * 100).round(2)
print(f"\nFormatet e eventDate: {len(t_formats)} formate, {int(is_range.sum())} intervale")
display(t_formats)

u = df.coordinateUncertaintyInMeters
t_unc = u.value_counts().head(10).rename("regjistrime").to_frame()
t_unc["perqindja_e_vlerave_te_dhena"] = (t_unc.regjistrime / u.notna().sum() * 100).round(2)
t_unc["publikuesi_kryesor"] = [publisher[u == v].mode().iat[0] for v in t_unc.index]
t_unc.index.name = "coordinateUncertaintyInMeters"
g7071 = df_raw[u == GRID_10KM]
print(f"\nVlerat më të shpeshta të coordinateUncertaintyInMeters ({int(u.notna().sum()):,} vlera të dhëna)")
display(t_unc)
print(f"7071 m: {len(g7071):,} regjistrime ({len(g7071) / u.notna().sum() * 100:.2f}% e vlerave të dhëna), "
      f"vetëm {g7071[['decimalLatitude', 'decimalLongitude']].drop_duplicates().shape[0]} pika të ndryshme; "
      f"{g7071.issue.fillna('').str.contains('COORDINATE_ROUNDED').mean() * 100:.2f}% kanë COORDINATE_ROUNDED")
display(publisher[g7071.index].value_counts().rename_axis("publikuesi").rename("regjistrime_me_7071").to_frame())

elev_bins = pd.cut(df.elevation.astype("float"), [-np.inf, 0, ELEV[0], 801, 1501, ELEV[1] + 1, np.inf], right=False,
                   labels=["< 0", "1 – 296", "297 – 800", "801 – 1500", "1501 – 2656", "> 2656"])
elev_bins = elev_bins.cat.add_categories("= 0")
elev_bins[df.elevation == 0] = "= 0"
t_elev = elev_bins.value_counts().reindex(["= 0", "< 0", "1 – 296", "297 – 800", "801 – 1500", "1501 – 2656",
                                           "> 2656"]).rename("regjistrime").to_frame()
t_elev["perqindja"] = (t_elev.regjistrime / df.elevation.notna().sum() * 100).round(2)
elev_out = df_raw.loc[elev & ~df.elevation.between(*ELEV),
                      ["gbifID", "elevation", "decimalLatitude", "decimalLongitude", "municipality", "basisOfRecord"]]
elev_out["publikuesi"] = publisher[elev_out.index]
elev_out = elev_out.sort_values("elevation").set_index("gbifID")
print(f"\nelevation: {int(df.elevation.notna().sum()):,} vlera të dhëna, min {df.elevation.min():g}, "
      f"max {df.elevation.max():g}")
display(t_elev)
print("Lartësitë jashtë 297–2656 m (pa 0):")
display(elev_out)


3.9 VLEFSHMËRIA


Tabela: Rregulla | Rreshta që dështojnë | %


Rregulla,Rreshta që dështojnë,%
"VL1 – Koordinatat brenda kutisë së Kosovës (lat 41.85–43.27, lon 20.01–21.80)",0,0.00
VL2 – Pika ka komunë të Kosovës,0,0.00
VL3 – Pa flamuj GBIF për koordinata të gabuara,0,0.00
VL4 – Koordinata me të paktën 3 shifra dhjetore (≤ ~110 m),784,1.67
VL5 – Viti brenda 1758–2026,0,0.00
VL6 – Viti ≥ 1900 (jo regjistrim historik),16,0.04
VL7 – month 1–12,0,0.00
VL8 – day 1–31,0,0.00
VL9 – year-month-day formon datë reale (p.sh. jo 30 shkurt),0,0.00
VL10 – eventDate lexohet si ISO 8601,0,0.00


% = rreshta që dështojnë / rreshta ku rregulla aplikohet (ku vlera ekziston) × 100

Detajet e kontrolleve


,korniza,kontrolli,kolonat,aplikohen,te_pavlefshme,perqindja_e_vlefshme,veprimi,perqindja_qe_deshtojne
id,,,,,,,,
VL1,V1,Koordinatat brenda kutisë së Kosovës (lat 41.8...,"decimalLatitude, decimalLongitude",46831,0,100.00,largo rreshtin,0.00
VL2,V1,Pika ka komunë të Kosovës,"municipality, decimalLatitude, decimalLongitude",46831,0,100.00,largo rreshtin,0.00
VL3,V1,Pa flamuj GBIF për koordinata të gabuara,issue,46831,0,100.00,largo rreshtin,0.00
VL4,–,Koordinata me të paktën 3 shifra dhjetore (≤ ~...,decimalLatitude,46831,784,98.33,shëno; jo për analizë në nivel fshati,1.67
VL5,V5,Viti brenda 1758–2026,year,44563,0,100.00,vendos NaN,0.00
VL6,A2,Viti ≥ 1900 (jo regjistrim historik),year,44563,16,99.96,shëno si historik,0.04
VL7,V6,month 1–12,month,43964,0,100.00,vendos NaN,0.00
VL8,V6,day 1–31,day,43612,0,100.00,vendos NaN,0.00
VL9,V6,year-month-day formon datë reale (p.sh. jo 30 ...,"year, month, day",43612,0,100.00,vendos NaN,0.00


...

,regjistrime_me_7071
publikuesi,
HabiProt,8122
"National Museum of Natural History, Bulgarian Academy of Sciences",102
"Botanical Garden, University of Valencia",1



elevation: 9,820 vlera të dhëna, min 0, max 3050


,regjistrime,perqindja
elevation,,
= 0,3634,37.01
< 0,0,0.00
1 – 296,4,0.04
297 – 800,3713,37.81
801 – 1500,1330,13.54
1501 – 2656,1134,11.55
> 2656,5,0.05


Lartësitë jashtë 297–2656 m (pa 0):


,elevation,decimalLatitude,decimalLongitude,municipality,basisOfRecord,publikuesi
gbifID,,,,,,
1837879632,100.0,42.666570,21.172400,Prishtinë / Priština,PRESERVED_SPECIMEN,"Museum of Comparative Zoology, Harvard University"
41659662,198.0,42.200000,20.716670,Prizren,OCCURRENCE,USDA Agricultural Research Service (ARS)
5284801271,250.0,42.675509,20.328745,Pejë / Peć,HUMAN_OBSERVATION,HabiProt
5284610427,293.0,42.226778,20.697054,Prizren,HUMAN_OBSERVATION,HabiProt
5794847480,2750.0,42.206670,21.119170,Kaçanik / Kačanik,PRESERVED_SPECIMEN,University of Vienna - Herbarium WU
2028433750,2800.0,42.206670,21.119170,Kaçanik / Kačanik,PRESERVED_SPECIMEN,University of Vienna - Herbarium WU
1830967944,2800.0,42.206670,21.119170,Kaçanik / Kačanik,PRESERVED_SPECIMEN,University of Vienna - Herbarium WU
1831004196,2800.0,42.206670,21.119170,Kaçanik / Kačanik,PRESERVED_SPECIMEN,University of Vienna - Herbarium WU
1830967907,3050.0,42.206670,21.119170,Kaçanik / Kačanik,PRESERVED_SPECIMEN,University of Vienna - Herbarium WU


## 4. RUAJTJA E RAPORTEVE

In [14]:
section("4. RUAJTJA E RAPORTEVE")
REPORTS_DIR.mkdir(exist_ok=True)
missing.to_csv(REPORTS_DIR / "01_vlerat_e_zbrazeta.csv")
quality.to_csv(REPORTS_DIR / "01_kontrollet_e_kualitetit.csv")
issues.to_csv(REPORTS_DIR / "01_gbif_issues.csv")
type_report.to_csv(REPORTS_DIR / "01_tipet_e_kolonave.csv")
framework.to_csv(REPORTS_DIR / "01_korniza_e_kualitetit.csv")
col_completeness.to_csv(REPORTS_DIR / "01_plotesia_kolona.csv")
row_completeness.to_csv(REPORTS_DIR / "01_plotesia_rreshta.csv")
validity.to_csv(REPORTS_DIR / "01_vlefshmeria.csv")
rule_table.to_csv(REPORTS_DIR / "01_vlefshmeria_rregullat.csv", index=False)
t_formats.to_csv(REPORTS_DIR / "01_eventdate_formatet.csv")
t_unc.to_csv(REPORTS_DIR / "01_pasiguria_vlerat.csv")
elev_out.to_csv(REPORTS_DIR / "01_lartesia_jashte_diapazonit.csv")
print("Raportet u ruajtën në", REPORTS_DIR.resolve())


4. RUAJTJA E RAPORTEVE
Raportet u ruajtën në C:\Master\PVDH\reports
